# Imports

In [1]:
import logging, warnings; logging.getLogger().setLevel(logging.ERROR);
warnings.filterwarnings("ignore")

import scanpy as sc
import scanpy.external as sce
import numpy as np
import pandas as pd
import re
from pathlib import Path 

import warnings, scipy.sparse as sp, matplotlib, matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.pyplot import rc_context
import matplotlib.font_manager
import matplotlib.lines as lines


pd.set_option('display.max_rows', 200)

matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42
matplotlib.rcParams['font.family'] = 'sans-serif'
matplotlib.rcParams['font.sans-serif'] = 'Arial'
matplotlib.rc('font', size=12)

sc.settings.n_jobs=-1
sc.set_figure_params(dpi=80, dpi_save=300, color_map='Spectral_r', vector_friendly=True, transparent=True)
sc.settings.figdir = '../../1_outputs/0_figures'
sc.settings.verbosity = 1 # verbosity: errors (0), warnings (1), info (2), hints (3)
sc.logging.print_header()

%matplotlib inline 
%config InlineBackend.figure_format = 'retina'

In [2]:
def sanitize_sheet_name(name):
    return name.replace(':', '_')

In [3]:
pwd

'/Users/mkaur/projects/7_dc/1_pyzone/0_notebooks/0_scRNA'

In [4]:
file_outputs = '../../1_outputs/' 
h5ad = '../../1_outputs/1_h5ad/'
deg_outputs = '../../1_outputs/2_deg/'

## Read in final adata file

In [5]:
## read in h5ad 

adata = sc.read_h5ad(h5ad + "9_adata_final.h5ad")

# DEGS

In [6]:
cell_types_list = adata.obs['cell_type_umbrella'].unique().tolist()
cell_types_list

['T', 'B', 'ILC', 'NK', 'DC']

In [7]:
cell_types_list = adata.obs['cell_type_umbrella_subset'].unique().tolist()
cell_types_list

['CD4', 'DP', 'CD8', 'B', 'ILC', 'Treg', 'DN', 'NK', 'DC', 'Unconv T']

## DEA SS_DONOR VS SS_HOST

In [12]:
adata.obs['condition'].value_counts()

condition
D1_Resident         11606
D1_Recirculating    10453
SS_Resident          9937
SS_Recirculating     5634
Name: count, dtype: int64

In [8]:
adata.obs['cell_type_umbrella_subset']

GCGACTTAGGATCACC-1@DAY1_DONOR_2    CD4
TACGCAATCGCTCCGT-1@SS_DONOR         DP
GTGTTACGTTCGGCAC-1@SS_HOST         CD4
CCTGAATGTTCCGTAG-1@SS_DONOR         DP
CGCTGTTCAATTCAAG-1@SS_HOST         CD4
                                  ... 
AAGTCAGGTTCGGGTC-1@DAY1_DONOR_2    ILC
CCGCAATAGGCTCAGT-1@DAY1_HOST        DP
GGTATGTCAAGTGCCA-1@DAY1_HOST       ILC
GCCAAATCAAACCGGC-1@SS_HOST         CD4
AATGACTTCCGTTGCC-1@DAY1_HOST       ILC
Name: cell_type_umbrella_subset, Length: 37630, dtype: category
Categories (10, object): ['DN' < 'DP' < 'CD4' < 'CD8' ... 'DC' < 'NK' < 'ILC' < 'B']

In [9]:
writer = pd.ExcelWriter(deg_outputs + 'ss_umbrella_subset_recirculating_vs_ss_resident.xlsx', engine='xlsxwriter')

In [10]:
for cell_type in cell_types_list:
    # Subset to condition
    adata_subset = adata[adata.obs['cell_type_umbrella_subset'] == cell_type].copy()

    # Run wilcoxon
    sc.tl.rank_genes_groups(adata_subset, groupby='condition', groups=['SS_Recirculating'], reference='SS_Resident',method='wilcoxon', use_raw=False)

    # Extract results
    result = adata_subset.uns['rank_genes_groups']
    groups = result['names'].dtype.names
    df = pd.DataFrame(
        {group + '_' + key[:1]: result[key][group]
        for group in groups for key in ['names', 'scores', 'logfoldchanges', 'pvals_adj']})

    # Write each condition to its own sheet
    df.to_excel(writer, sheet_name=sanitize_sheet_name(cell_type))

writer.close()
 

## DEA DAY1_DONOR VS DAY1_HOST

In [11]:
writer = pd.ExcelWriter(deg_outputs + 'inj_umbrella_subset_recirculating_vs_inj_resident.xlsx', engine='xlsxwriter')

In [12]:
for cell_type in cell_types_list:
    # Subset to condition
    adata_subset = adata[adata.obs['cell_type_umbrella_subset'] == cell_type].copy()

    # Run wilcoxon
    sc.tl.rank_genes_groups(adata_subset, groupby='condition', groups=['D1_Recirculating'], reference='D1_Resident',method='wilcoxon', use_raw=False)

    # Extract results
    result = adata_subset.uns['rank_genes_groups']
    groups = result['names'].dtype.names
    df = pd.DataFrame(
        {group + '_' + key[:1]: result[key][group]
        for group in groups for key in ['names', 'scores', 'logfoldchanges', 'pvals_adj']})

    # Write each condition to its own sheet
    df.to_excel(writer, sheet_name=sanitize_sheet_name(cell_type))

writer.close()
 

## DEA SS_HOST VS SS_DONOR

In [12]:
adata.obs['condition'].value_counts()

condition
D1_Resident         11606
D1_Recirculating    10453
SS_Resident          9937
SS_Recirculating     5634
Name: count, dtype: int64

In [13]:
writer = pd.ExcelWriter(deg_outputs + 'ss_resident_vs_ss_recirculating.xlsx', engine='xlsxwriter')

In [14]:
for cell_type in cell_types_list:
    # Subset to condition
    adata_subset = adata[adata.obs['cell_type_umbrella'] == cell_type].copy()

    # Run wilcoxon
    sc.tl.rank_genes_groups(adata_subset, groupby='condition', groups=['SS_Resident'], reference='SS_Recirculating',method='wilcoxon', use_raw=False)

    # Extract results
    result = adata_subset.uns['rank_genes_groups']
    groups = result['names'].dtype.names
    df = pd.DataFrame(
        {group + '_' + key[:1]: result[key][group]
        for group in groups for key in ['names', 'scores', 'logfoldchanges', 'pvals_adj']})

    # Write each condition to its own sheet
    df.to_excel(writer, sheet_name=sanitize_sheet_name(cell_type))

writer.close()
 

## DEA DAY1_HOST VS DAY1_DONOR 

In [15]:
writer = pd.ExcelWriter(deg_outputs + 'inj_resident_vs_inj_recirculating.xlsx', engine='xlsxwriter')

In [ ]:
for cell_type in cell_types_list:
    # Subset to condition
    adata_subset = adata[adata.obs['cell_type_umbrella'] == cell_type].copy()

    # Run wilcoxon
    sc.tl.rank_genes_groups(adata_subset, groupby='condition', groups=['D1_Resident'], reference='D1_Recirculating',method='wilcoxon', use_raw=False)

    # Extract results
    result = adata_subset.uns['rank_genes_groups']
    groups = result['names'].dtype.names
    df = pd.DataFrame(
        {group + '_' + key[:1]: result[key][group]
        for group in groups for key in ['names', 'scores', 'logfoldchanges', 'pvals_adj']})

    # Write each condition to its own sheet
    df.to_excel(writer, sheet_name=sanitize_sheet_name(cell_type))

writer.close()
 

## D1 Recirculating vs D1 Resident

In [6]:
writer = pd.ExcelWriter(deg_outputs + 'd1_recirculating_vs_resident.xlsx', engine='xlsxwriter')

In [8]:
adata_subset = adata[adata.obs['condition'].isin(['D1_Recirculating', 'D1_Resident'])].copy()

# Run wilcoxon
sc.tl.rank_genes_groups(adata_subset, groupby='condition', groups=['D1_Recirculating'], reference='D1_Resident',method='wilcoxon', use_raw=False)

# Extract results
result = adata_subset.uns['rank_genes_groups']
groups = result['names'].dtype.names
df = pd.DataFrame(
    {group + '_' + key[:1]: result[key][group]
    for group in groups for key in ['names', 'scores', 'logfoldchanges', 'pvals_adj']})

# Write each condition to its own sheet
df.to_excel(writer)

writer.close()
 

## D1 Resident vs D1 Recirculating

In [22]:
writer = pd.ExcelWriter(deg_outputs + 'd1_resident_vs_recirculating.xlsx', engine='xlsxwriter')

In [23]:
adata_subset = adata[adata.obs['condition'].isin(['D1_Recirculating', 'D1_Resident'])].copy()

# Run wilcoxon
sc.tl.rank_genes_groups(adata_subset, groupby='condition', groups=['D1_Resident'], reference='D1_Recirculating',method='wilcoxon', use_raw=False)

# Extract results
result = adata_subset.uns['rank_genes_groups']
groups = result['names'].dtype.names
df = pd.DataFrame(
    {group + '_' + key[:1]: result[key][group]
    for group in groups for key in ['names', 'scores', 'logfoldchanges', 'pvals_adj']})

# Write each condition to its own sheet
df.to_excel(writer)

writer.close()
 

## Combine D1 and SS Conditions

In [29]:
writer = pd.ExcelWriter(deg_outputs + 'd1_ss_combo_cell_type_umbrella_recirculating_vs_resident.xlsx', engine='xlsxwriter')

In [26]:
adata.obs

,sample,condition,day,source,cell_type_subset,cell_type_umbrella,umbrella_condition,subset_condition,cell_type_umbrella_subset
GCGACTTAGGATCACC-1@DAY1_DONOR_2,DAY1_DONOR_2,D1_Recirculating,D1,Recirculating,T:CD4,T,T_D1_Recirculating,T:CD4_D1_Recirculating,CD4
TACGCAATCGCTCCGT-1@SS_DONOR,SS_DONOR,SS_Recirculating,Steady State,Recirculating,T:DP(Q),T,T_SS_Recirculating,T:DP(Q)_SS_Recirculating,DP
GTGTTACGTTCGGCAC-1@SS_HOST,SS_HOST,SS_Resident,Steady State,Resident,T:CD4 Immature,T,T_SS_Resident,T:CD4 Immature_SS_Resident,CD4
CCTGAATGTTCCGTAG-1@SS_DONOR,SS_DONOR,SS_Recirculating,Steady State,Recirculating,T:DP(Q),T,T_SS_Recirculating,T:DP(Q)_SS_Recirculating,DP
CGCTGTTCAATTCAAG-1@SS_HOST,SS_HOST,SS_Resident,Steady State,Resident,T:CD4 Immature,T,T_SS_Resident,T:CD4 Immature_SS_Resident,CD4
...,...,...,...,...,...,...,...,...,...
AAGTCAGGTTCGGGTC-1@DAY1_DONOR_2,DAY1_DONOR_2,D1_Recirculating,D1,Recirculating,ILC:ILCP,ILC,ILC_D1_Recirculating,ILC:ILCP_D1_Recirculating,ILC
CCGCAATAGGCTCAGT-1@DAY1_HOST,DAY1_HOST,D1_Resident,D1,Resident,T:DP(Q),T,T_D1_Resident,T:DP(Q)_D1_Resident,DP
GGTATGTCAAGTGCCA-1@DAY1_HOST,DAY1_HOST,D1_Resident,D1,Resident,ILC:ILCP,ILC,ILC_D1_Resident,ILC:ILCP_D1_Resident,ILC
GCCAAATCAAACCGGC-1@SS_HOST,SS_HOST,SS_Resident,Steady State,Resident,T:CD4 Immature,T,T_SS_Resident,T:CD4 Immature_SS_Resident,CD4


In [ ]:
adata.obs['cell_type_umbrella_source'] = adata.obs[""]

In [ ]:
for cell_type in cell_types_list:
    # Subset to condition
    adata_subset = adata[adata.obs['cell_type_umbrella'] == cell_type].copy()

    # Run wilcoxon
    sc.tl.rank_genes_groups(adata_subset, groupby='source', groups=['Recirculating'], reference='Resident', method='wilcoxon', use_raw=False)

    # Extract results
    result = adata_subset.uns['rank_genes_groups']
    groups = result['names'].dtype.names
    df = pd.DataFrame(
        {group + '_' + key[:1]: result[key][group]
        for group in groups for key in ['names', 'scores', 'logfoldchanges', 'pvals_adj']})

    # Write each condition to its own sheet
    df.to_excel(writer, sheet_name=sanitize_sheet_name(cell_type))

writer.close()
 